In [1]:
#Imports
import time
import json
import os
import subprocess
from datetime import datetime
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage

In [2]:
# 1. Setup & Ollama Verbindung
windows_ip = subprocess.check_output("ip route list default | awk '{print $3}'", shell=True).decode('utf-8').strip()
model = ChatOllama(model="qwen2.5:3b", base_url=f"http://{windows_ip}:11434", temperature=0.0, )

# 2. Verbesserte Tracking-Funktion, die Datenpakete (Dictionaries) zurückgibt
def run_tracked_prompt(prompt_text, step_name, max_tokens):
    print(f"--- Excecuting: {step_name} (Hard Limit: {max_tokens} Output Tokens) ---")    
    start_time = time.time()
    
    bound_model = model.bind(num_predict=max_tokens)
    response = bound_model.invoke([HumanMessage(content=prompt_text)])
    
    total_time = round(time.time() - start_time, 2)
    meta = response.response_metadata
    in_tokens = meta.get('prompt_eval_count', 0)
    out_tokens = meta.get('eval_count', 0)
    
    print(f"Done after {total_time}s | Tokens: {in_tokens} In / {out_tokens} Out")
    
    # Anstatt nur Text geben wir jetzt ein strukturiertes Datenpaket zurück
    return {
        "step_name": step_name,
        "prompt": prompt_text,
        "output": response.content,
        "latency_seconds": total_time,
        "input_tokens": in_tokens,
        "output_tokens": out_tokens
    }



/tmp/ipykernel_28847/1004709605.py:3: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  model = ChatOllama(model="qwen2.5:3b", base_url=f"http://{windows_ip}:11434", temperature=0.0, )


In [3]:
# ==========================================
# DIE PIPELINE & DAS LOGGING
# ==========================================

# 1. Daten sauber trennen
context_text = "Exactly six trade representatives negotiate a treaty: Klosnik, Londi, Manley, Neri, Osata, Poirier. There are exactly six chairs evenly spaced around a circular table. The chairs are numbered 1 through 6, with successively numbered chairs next to each other and chair number 1 next to chair number 6. Each chair is occupied by exactly one of the representatives. The following conditions apply: Poirier sits immediately next to Neri. Londi sits immediately next to Manley, Neri, or both. Klosnik does not sit immediately next to Manley. If Osata sits immediately next to Poirier, Osata does not sit immediately next to Manley."

# Angepasst, da die Multiple-Choice Optionen fehlen
question_text = "Generate exactly one valid seating arrangement of the six representatives in chairs 1 through 6 that does NOT violate the stated conditions."

experiment_data = {
    "timestamp": datetime.now().isoformat(),
    "model": "qwen2.5:3b",
    "context": context_text,
    "question": question_text,
    "steps": []
}

In [4]:

# ---------------------------------------------------------
# Schritt 0: Baseline (Without Reasoning)
# Struktur: Task -> Context -> Question -> Task
# ---------------------------------------------------------
prompt_step0 = f"""Task: Solve the logic puzzle based on the context. You must ONLY output the final seating arrangement as a comma-separated list of the 6 names. Do NOT provide any explanations or reasoning.
Context: {context_text}
Question: {question_text}
Task: ONLY output the sequence. Example format: Name1, Name2, Name3, Name4, Name5, Name6"""

step0_result = run_tracked_prompt(prompt_step0, "Step 0: Baseline", max_tokens=50)
experiment_data["steps"].append(step0_result)

--- Excecuting: Step 0: Baseline (Hard Limit: 50 Output Tokens) ---
Done after 31.52s | Tokens: 269 In / 20 Out


In [5]:
# ---------------------------------------------------------
# Schritt 1: Reasoning Tokens (Begrenzt)
# Struktur: Task -> Context -> Question -> Task
# ---------------------------------------------------------
# Die Begrenzung erfolgt durch die Anweisung "Concise step-by-step" und ein Limit.
prompt_step1 = f"""Task: Think step-by-step to solve the logic puzzle. Generate a concise logical deduction path. Limit your reasoning to a maximum of 500 words.
Context: {context_text}
Question: {question_text}
Task: Provide your concise step-by-step logical deduction. Do not output the final list yet, only the logical steps."""

step1_result = run_tracked_prompt(prompt_step1, "Step 1: Reasoning Generation", max_tokens=700)
experiment_data["steps"].append(step1_result)

--- Excecuting: Step 1: Reasoning Generation (Hard Limit: 700 Output Tokens) ---
Done after 177.84s | Tokens: 260 In / 700 Out


In [6]:
# ---------------------------------------------------------
# Schritt 2: Synthese
# Struktur: Task -> Context -> Question -> Task -> Reasoning -> Question -> Task
# ---------------------------------------------------------
prompt_step2 = f"""Task: Solve the logic puzzle using the provided thought process. You must ONLY output the final seating arrangement as a comma-separated list of the 6 names. Do NOT provide any explanations.
Context: {context_text}
Question: {question_text}
Task: Review the following thought process.
Thought Process: {step1_result['output']}
Question: {question_text}
Task: Based on the thought process, ONLY output the final sequence. Example format: Name1, Name2, Name3, Name4, Name5, Name6"""

step2_result = run_tracked_prompt(prompt_step2, "Step 2: Synthesis", max_tokens=50)
experiment_data["steps"].append(step2_result)

--- Excecuting: Step 2: Synthesis (Hard Limit: 50 Output Tokens) ---
Done after 114.25s | Tokens: 1013 In / 20 Out


In [7]:
# ---------------------------------------------------------
# 3. Daten in die JSON-Datei schreiben
# ---------------------------------------------------------
log_file = "experiment_log.json"

if os.path.exists(log_file):
    with open(log_file, "r", encoding="utf-8") as f:
        log_data = json.load(f)
else:
    log_data = []

log_data.append(experiment_data)

with open(log_file, "w", encoding="utf-8") as f:
    json.dump(log_data, f, indent=4, ensure_ascii=False)

print(f"\nExperiment successfully conducted and documented in: {log_file}")
print(f"--- Results ---")
print(f"Baseline Output: {step0_result['output']} in {step0_result['latency_seconds']}s")
print(f"Reasoning Generation: {step1_result['output']} in {step1_result['latency_seconds']}s")
print(f"Synthesis Output: {step2_result['output']} in {step2_result['latency_seconds']}s")


Experiment successfully conducted and documented in: experiment_log.json
--- Results ---
Baseline Output: Klosnik, Londi, Poirier, Neri, Manley, Osata in 31.52s
Reasoning Generation: To solve this logic puzzle, we will analyze each condition and deduce the seating arrangement step by step.

### Step 1: Analyze Poirier's Position
- **Condition**: Poirier sits immediately next to Neri.
- This means if Poirier is in chair 1, then Neri must be in chair 2 or chair 6. If Poirier is in chair 2, then Neri must be in chair 1 or chair 3. Similarly for other chairs.

### Step 2: Analyze Londi's Position
- **Condition**: Londi sits immediately next to Manley, Neri, or both.
- This means if Londi and Manley are together (either adjacent or separated by one chair), then they must be seated in consecutive positions. If Londi is in chair 1, then Manley can only be in chair 2 or chair 6.

### Step 3: Analyze Klosnik's Position
- **Condition**: Klosnik does not sit immediately next to Manley.
- This me

In [8]:
# ---------------------------------------------------------
# Schritt 3: Validierung gegen Benchmark
# ---------------------------------------------------------

benchmark_results = [
    "Klosnik, Poirier, Neri, Manley, Osata, Londi",
    "Klosnik, Londi, Manley, Poirier, Neri, Osata",
    "Klosnik, Londi, Manley, Osata, Poirier, Neri",
    "Klosnik, Osata, Poirier, Neri, Londi, Manley",
    "Klosnik, Neri, Londi, Osata, Manley, Poirier"
]

# Extract outputs from experiment steps
baseline_output = step0_result['output'].strip()
synthesis_output = step2_result['output'].strip()

# Compare each result with benchmark
results_comparison = {
    "baseline_match": baseline_output in benchmark_results,
    "synthesis_match": synthesis_output in benchmark_results,
    "baseline_output": baseline_output,
    "synthesis_output": synthesis_output,
    "benchmark_solutions": benchmark_results
}

print("\n--- Validation Results ---")
print(f"Baseline matches benchmark: {results_comparison['baseline_match']}")
print(f"Synthesis matches benchmark: {results_comparison['synthesis_match']}")
print(f"\nBaseline result: {baseline_output}")
print(f"Synthesis result: {synthesis_output}")

# Add comparison to experiment data
experiment_data["validation"] = results_comparison


--- Validation Results ---
Baseline matches benchmark: False
Synthesis matches benchmark: False

Baseline result: Klosnik, Londi, Poirier, Neri, Manley, Osata
Synthesis result: Poirier, Londi, Manley, Neri, Osata, Klosnik
